# Bonus — Multi-Agent Supervisor Pattern (Software Team) · Google Colab

> **This is a backup / bonus notebook.** The main Day-1 finale is
> `Exercise_3_MultiAgent_Colab.ipynb` (a Research + Verifier RAG team over your **`story-llama`**
> document). This notebook shows the **exact same Supervisor pattern** applied to a different
> domain — a software-engineer + QA-engineer coding team — to prove the pattern generalizes
> beyond RAG.

### Why keep it
The orchestration skeleton is *identical* to the main Exercise 3:

```
Start → Supervisor → Condition
                        ├─ worker A → (loop back to Supervisor)
                        ├─ worker B → (loop back to Supervisor)
                        └─ FINISH → final report (end)
```

Only the **workers and the Supervisor's rules** change. In the main exercise the workers are
`research_agent` + `verifier_agent`; here they are `software_engineer` + `QA_engineer`.
Same shared `flow_state`, same conversation memory, same JSON routing, same loop caps.

### Prerequisites
- Colab secrets (🔑): **either** `GOOGLE_API_KEY` or `OPENAI_API_KEY`
- Set `LLM_PROVIDER` in Step 1 (`"google"` or `"openai"`)
- *No Pinecone needed — this bonus has no RAG.*


## Step 0 — Install
Pure LLM orchestration, no vector DB.


In [1]:
%pip install -q langchain langchain-core langchain-google-genai google-generativeai langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 20.3 MB/s eta 0:00:00


## Step 1 — Load API key & pick LLM provider
Add `GOOGLE_API_KEY` or `OPENAI_API_KEY` in 🔑, then set `LLM_PROVIDER` below.


In [2]:
LLM_PROVIDER = "google"  # "google" | "openai"

from google.colab import userdata
import os

if LLM_PROVIDER == "google":
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
elif LLM_PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    raise ValueError('LLM_PROVIDER must be "google" or "openai"')

print(f"Keys loaded · LLM provider: {LLM_PROVIDER}")

Keys loaded · LLM provider: google


## Step 2 — Setup
LLM helpers and shared state in one cell.


In [31]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import json
import re # Import the regular expression module

TEMP = 0.4

def get_llm(temperature=TEMP):
    if LLM_PROVIDER == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model="gemma-4-31b-it", temperature=temperature)
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)

def to_text(content) -> str:
    if isinstance(content, str): return content
    if isinstance(content, dict): return content.get("text", str(content))
    if isinstance(content, list): return "".join(to_text(p) for p in content)
    return str(content)

def ask_llm(llm, messages) -> str:
    return to_text(llm.invoke(messages).content)

def parse_json_object(text: str) -> dict:
    text = text.strip()
    original_input_text = text
    parsed_objects = []
    decoder = json.JSONDecoder()

    # 1. Handle markdown code blocks first
    if "```" in text:
        parts = text.split("```")
        for part in parts:
            trimmed_part = part.strip()
            if trimmed_part.startswith("{") and trimmed_part.endswith("}"):
                text_to_parse = trimmed_part.removeprefix("json").strip()
                try:
                    parsed_objects.append(json.loads(text_to_parse))
                except json.JSONDecodeError:
                    pass
        if parsed_objects:
            return parsed_objects[-1]
        text = text.replace("```json", "").replace("```", "").strip() # Clean text for further attempts

    # 2. Iteratively find and parse JSON objects from the remaining text
    current_text_to_search = text
    current_offset = 0 # Offset in the original `text` string

    while current_offset < len(current_text_to_search):
        # Find the next potential start of a JSON object
        start_brace_relative_idx = current_text_to_search.find('{', current_offset)

        if start_brace_relative_idx == -1:
            break # No more '{' found in the remaining text, stop

        # Slice the string from the potential start of JSON
        potential_json_segment = current_text_to_search[start_brace_relative_idx:]

        try:
            # Attempt to decode a JSON object from this segment
            obj, end_pos_in_segment = decoder.raw_decode(potential_json_segment)
            parsed_objects.append(obj)
            # Advance offset past the successfully parsed object
            current_offset = start_brace_relative_idx + end_pos_in_segment
        except json.JSONDecodeError:
            # If standard raw_decode fails, try to fix this segment and parse again
            fixed_segment = potential_json_segment

            # Apply robust fixes: unquoted keys, single quotes, trailing commas
            fixed_segment = re.sub(r'([{,][ \t\n\r\f\v]*)(\w+)([ \t\n\r\f\v]*:)', r'\1"\2"\3', fixed_segment)
            fixed_segment = re.sub(r"'(.*?)'", r'"\1"', fixed_segment, flags=re.DOTALL)
            fixed_segment = re.sub(r',\s*([}\]])', r'\1', fixed_segment)

            try:
                obj, end_pos_in_fixed_segment = decoder.raw_decode(fixed_segment)
                parsed_objects.append(obj)
                # If a fixed segment was parsed, it's safer to advance the offset past where the original '{' was,
                # and let the loop find the next potential JSON from there.
                current_offset = start_brace_relative_idx + 1
            except json.JSONDecodeError:
                # If even fixing the segment and re-attempting raw_decode fails,
                # then this segment is truly problematic. Just advance past the current '{'.
                current_offset = start_brace_relative_idx + 1

    if parsed_objects:
        return parsed_objects[-1] # Return the last successfully parsed JSON object
    else:
        # If no objects were parsed after all attempts
        raise ValueError(f"Failed to parse JSON. Original input: '{original_input_text}'. No valid JSON object could be extracted or parsed.")

flow_state = {"next": "", "instructions": ""}
conversation_history = []

## Step 3 — Supervisor


In [32]:
SUPERVISOR_SYSTEM = """You are a supervisor managing:
- software_engineer
- QA_engineer

Rules:
1. First assign task to software_engineer
2. After software_engineer completes, ALWAYS assign QA_engineer
3. After QA_engineer review, respond with FINISH
4. Do NOT skip QA_engineer
5. Do NOT repeat same worker unnecessarily

Return ONLY valid JSON with exactly these two keys:
{"next": "software_engineer" | "QA_engineer" | "FINISH", "instructions": "task details here"}
No extra text. No markdown. Just JSON."""

def run_supervisor():
    raw = ask_llm(get_llm(), [SystemMessage(content=SUPERVISOR_SYSTEM)] + conversation_history)
    output = parse_json_object(raw)
    flow_state["next"], flow_state["instructions"] = output["next"], output["instructions"]
    conversation_history.append(AIMessage(content=raw))
    print(f"\n[Supervisor] -> {output['next']}: {output['instructions'][:120]}...")
    return output

## Step 4 — Condition router


In [33]:
def check_condition():
    nxt = flow_state["next"]
    if nxt == "software_engineer": return "software_engineer"
    if nxt == "QA_engineer": return "QA_engineer"
    return "FINISH"

## Step 5 — Workers


In [34]:
SE_SYSTEM = "You are a software developer. Follow the supervisor's instructions."
QA_SYSTEM = """As a QA Engineer, review the code from the Software Engineer.
Check correctness, edge cases, and readability. Provide constructive feedback."""

def run_software_engineer():
    task = f"Supervisor instructions:\n{flow_state['instructions']}"
    output = ask_llm(get_llm(), [SystemMessage(content=SE_SYSTEM)] + conversation_history + [HumanMessage(content=task)])
    conversation_history.append(HumanMessage(content=output))
    print("\n[software_engineer]", output[:400], "...")
    return output

def run_qa_engineer():
    task = f"Supervisor instructions:\n{flow_state['instructions']}"
    output = ask_llm(get_llm(), [SystemMessage(content=QA_SYSTEM)] + conversation_history + [HumanMessage(content=task)])
    conversation_history.append(HumanMessage(content=output))
    print("\n[QA_engineer]", output[:400], "...")
    return output

## Step 6 — FINISH


In [35]:
FINISH_SYSTEM = """Document the complete multi-agent workflow: user request, each agent's work,
code produced, QA feedback, and final outcome."""

def run_finish():
    task = f"Supervisor instructions:\n{flow_state['instructions']}"
    output = ask_llm(get_llm(), [SystemMessage(content=FINISH_SYSTEM)] + conversation_history + [HumanMessage(content=task)])
    print("\n" + "=" * 60 + "\n[FINISH]\n" + "=" * 60)
    print(output)
    return output

## Step 7 — Orchestrator


In [36]:
def run_flow(user_request: str, max_loops: int = 5):
    global conversation_history, flow_state
    conversation_history, flow_state = [], {"next": "", "instructions": ""}
    print("=" * 60 + f"\nUSER REQUEST: {user_request}\n" + "=" * 60)
    conversation_history.append(HumanMessage(content=user_request))
    loop_count = {"software_engineer": 0, "QA_engineer": 0}

    while True:
        run_supervisor()
        route = check_condition()
        print(f"\n[Condition] -> {route}")

        if route == "software_engineer":
            if loop_count["software_engineer"] >= max_loops:
                flow_state["next"] = "FINISH"; continue
            run_software_engineer()
            loop_count["software_engineer"] += 1
        elif route == "QA_engineer":
            if loop_count["QA_engineer"] >= max_loops:
                flow_state["next"] = "FINISH"; continue
            run_qa_engineer()
            loop_count["QA_engineer"] += 1
        elif route == "FINISH":
            result = run_finish()
            print("\n✅ FLOW COMPLETE")
            return result

## Step 8 — Run


In [37]:
result = run_flow("Build me a login system with username and password validation in Python", max_loops=5)

USER REQUEST: Build me a login system with username and password validation in Python

[Supervisor] -> software_engineer: Implement a login system in Python that includes username and password validation. Ensure it handles basic user registra...

[Condition] -> software_engineer

[software_engineer] {'type': 'thinking', 'thinking': "The objective is to build a login system in Python with username and password validation, including registration and authentication.\n\n    *   User storage (for a simple example, a dictionary or a JSON file).\n    *   Registration function: Check if the user already exists, validate password strength (optional but good), and store credentials.\n    *   Login func ...

[Supervisor] -> QA_engineer: Review the provided Python login system. Verify that the registration and login logic works as expected, check that pass...

[Condition] -> QA_engineer

[QA_engineer] {'type': 'thinking', 'thinking': 'QA Engineer.\nReview a Python login system implemented by a Sof

## Step 9 — 🔧 Your turn
Change the `user_request` below and re-run. Compare with `Exercise_3_MultiAgent_Colab.ipynb` — same skeleton, different workers.


In [ ]:
result = run_flow("Write a Python function to check if a string is a palindrome")

## ✅ Takeaway
The **Supervisor pattern is domain-agnostic**. Swap the workers and rewrite the rules, and the
same orchestration runs a research team, a coding team, or any other pipeline of specialists.

| | Main Exercise 3 (RAG) | This bonus (Software team) |
|---|---|---|
| Worker A | `research_agent` (uses `document_search`) | `software_engineer` |
| Worker B | `verifier_agent` (groundedness) | `QA_engineer` |
| Supervisor / Condition / FINISH | identical | identical |

Back to the main track: `Exercise_3_MultiAgent_Colab.ipynb`, then on to Day 2.
